<a href="https://colab.research.google.com/github/ProjectsDataWill/tech-challenge-olist-sla/blob/main/Tech_Challenge_Fase_1_Willian_Almeida.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Instalar (se necessário)
!pip install kagglehub[pandas-datasets]

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Função helper
def load_table(file_path):
    return kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "olistbr/brazilian-ecommerce",
        file_path
    )

# Carregar tabelas principais
orders = load_table("olist_orders_dataset.csv")
order_items = load_table("olist_order_items_dataset.csv")
reviews = load_table("olist_order_reviews_dataset.csv")
customers = load_table("olist_customers_dataset.csv")
sellers = load_table("olist_sellers_dataset.csv")

/tmp/ipykernel_1455/2714138024.py:13: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  return kagglehub.load_dataset(


Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.


/tmp/ipykernel_1455/2714138024.py:13: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  return kagglehub.load_dataset(


Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.


/tmp/ipykernel_1455/2714138024.py:13: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  return kagglehub.load_dataset(


Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.


/tmp/ipykernel_1455/2714138024.py:13: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  return kagglehub.load_dataset(


Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.


/tmp/ipykernel_1455/2714138024.py:13: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  return kagglehub.load_dataset(


Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.


In [3]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

## 1. Feature Engineering (SLA)

In [4]:
# Tempo real
orders['lead_time_total'] = (
    orders['order_delivered_customer_date'] -
    orders['order_purchase_timestamp']
).dt.days

# Tempo prometido
orders['estimated_lead_time'] = (
    orders['order_estimated_delivery_date'] -
    orders['order_purchase_timestamp']
).dt.days

# Delay
orders['delay'] = (
    orders['order_delivered_customer_date'] -
    orders['order_estimated_delivery_date']
).dt.days

# Flag atraso
orders['is_delayed'] = orders['delay'] > 0

# Gap de SLA (INSIGHT PRINCIPAL)
orders['sla_gap'] = orders['estimated_lead_time'] - orders['lead_time_total']

## 2. Distribuição de atraso (BRUTO)


In [16]:
fig = px.histogram(
    orders,
    x="delay",
    nbins=60,
    title="Distribuição de Atraso (Dados Brutos)"
)

fig.add_vline(x=0, line_dash="dash", line_color="red")



fig.show()

## 3. Distribuição de atraso (TRATADO)

In [14]:
orders_filtered = orders[
    (orders['delay'] > -30) &
    (orders['delay'] < 30)
]

fig = px.histogram(
    orders_filtered,
    x="delay",
    nbins=50,
    title="Distribuição de Atraso (Faixa Realista)"
)

fig.add_vline(x=0, line_dash="dash", line_color="red")

fig.update_layout(
    paper_bgcolor='white',
    plot_bgcolor='white',
    font=dict(color='black')
)

fig.show()

## 3. Prazo Prometido vs Real


In [7]:
fig = px.scatter(
    orders,
    x="estimated_lead_time",
    y="lead_time_total",
    opacity=0.3,
    title="Prazo Prometido vs Prazo Real"
)

fig.add_shape(
    type="line",
    x0=0, y0=0,
    x1=100, y1=100,
    line=dict(color="red", dash="dash")
)

fig.update_layout(
    paper_bgcolor='white',
    plot_bgcolor='white',
    font=dict(color='black')
)

fig.show()

## 4. Distribuição do erro de SLA

In [8]:
fig = px.histogram(
    orders,
    x="sla_gap",
    nbins=60,
    title="Diferença entre Prazo Prometido e Real"
)

fig.add_vline(x=0, line_dash="dash", line_color="red")

fig.show()

## 5. Classificação do SLA

In [9]:
def classify_sla(x):
    if x < -5:
        return "Atrasado"
    elif x <= 5:
        return "No prazo"
    else:
        return "Antecipado"

orders['sla_category'] = orders['sla_gap'].apply(classify_sla)

## 6. Distribuição das categorias

In [10]:
fig = px.pie(
    orders,
    names="sla_category",
    title="Classificação do SLA"
)

fig.show()

## 7. Impacto no cliente (SLA vs Review)

In [17]:
df = orders.merge(reviews, on="order_id", how="left")

# Trocar True/False por rótulos mais claros
df["status_entrega"] = df["is_delayed"].map({
    False: "No prazo",
    True: "Atrasado"
})

fig = px.box(
    df,
    x="status_entrega",
    y="review_score",
    title="Avaliação do Cliente por Status de Entrega",
    labels={
        "status_entrega": "Status da entrega",
        "review_score": "Nota da avaliação"
    }
)

fig.update_layout(
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(color="black")
)

fig.show()

## 8. Tempo de entrega vs nota

In [12]:
fig = px.box(
    df,
    x="review_score",
    y="lead_time_total",
    title="Tempo de Entrega por Nota"
)

fig.show()

## 9. Análise por estado

In [22]:

import plotly.graph_objects as go

# Base geográfica
df_geo = df.merge(customers, on="customer_id", how="left")

# Agrupamento por estado
sla_state = (
    df_geo.groupby("customer_state", as_index=False)
    .agg(
        total_orders=("order_id", "count"),
        delayed=("is_delayed", "sum")
    )
)

# Taxa de atraso
sla_state["delay_rate"] = (
    sla_state["delayed"] / sla_state["total_orders"] * 100
)

# Ordenação por maior atraso
sla_state_sorted = sla_state.sort_values("delay_rate", ascending=False).copy()

# Rótulos
sla_state_sorted["delay_rate_str"] = (
    sla_state_sorted["delay_rate"].round(1).astype(str) + "%"
)

fig = go.Figure()

# Barras: % de atraso
fig.add_trace(
    go.Bar(
        x=sla_state_sorted["customer_state"],
        y=sla_state_sorted["delay_rate"],
        text=sla_state_sorted["delay_rate_str"],
        textposition="outside",
        name="% de atraso",
        marker=dict(color="#5B6CFF"),
        hovertemplate=(
            "<b>Estado:</b> %{x}<br>"
            "<b>% de atraso:</b> %{y:.1f}%<br>"
            "<extra></extra>"
        )
    )
)

# Linha: volume total de pedidos
fig.add_trace(
    go.Scatter(
        x=sla_state_sorted["customer_state"],
        y=sla_state_sorted["total_orders"],
        mode="lines+markers",
        name="Total de pedidos",
        yaxis="y2",
        line=dict(color="#FF4B3E", width=2),
        marker=dict(size=6),
        opacity=0.75,
        hovertemplate=(
            "<b>Estado:</b> %{x}<br>"
            "<b>Total de pedidos:</b> %{y}<br>"
            "<extra></extra>"
        )
    )
)

fig.update_layout(
    title="% de Pedidos Atrasados por Estado vs Volume de Pedidos",
    xaxis=dict(
        title="Estado",
        tickangle=0
    ),
    yaxis=dict(
        title="% de atraso",
        ticksuffix="%",
        range=[0, sla_state_sorted["delay_rate"].max() * 1.25]
    ),
    yaxis2=dict(
        title="Total de pedidos",
        overlaying="y",
        side="right",
        showgrid=False
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.08,
        xanchor="center",
        x=0.5,
        font=dict(size=12)
    ),
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(color="black", size=12),
    margin=dict(t=90, b=60, l=70, r=80),
    height=500
)

fig.show()